# Supplementary Figure 1 | Annotation change and quantification impact from GTEx v10 to v11

**a.** Number of transcripts annotated for each gene biotype in the GENCODE reference used by GTEx releases over time. Shaded area is the previous GTEx release (v10) to this release (v11). lncRNA transcript annotations increase from 53,009 in v10 to 191,106 in v11.

**b.** Numbers of genes with gene-, transcript-, or exon-level annotation changes between GENCODE v39 and v47, stratified by change class (renamed gene; removed, renamed, moved, or new transcript/exon) and stacked by v47 biotype. Transcript and exon bars are among genes with modified or new transcripts; renamed classes include genes with reassigned identifiers.

**c.** Percentage of STAR-assigned reads per tissue mapping to each biotype or STAR category (ambiguous strand, multimapping, unannotated, unmapped) in GTEx v10 versus v11.

**d.** Tissue-level change (v11 − v10) in the percentage of reads assigned to lncRNA genes, stratified by pooled gene-change categories (removed gene; no change; modified gene, pooling modified transcript, new transcript, and reassigned gene; new gene). New lncRNA genes account for most of the gain in lncRNA read share.

**e.** Tissue-level change (v11 − v10) in the percentage of reads assigned to protein-coding genes, stratified by the increase in each gene’s overlapping/ambiguous sequence fraction from v39 to v47 (genes with decreased ambiguity excluded). Larger ambiguity increases are associated with larger decreases in protein-coding read share.


## Setup


In [ ]:
from pathlib import Path

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm

plt.rc("axes", labelsize=6)
plt.rc("xtick", labelsize=6)
plt.rc("ytick", labelsize=6)
plt.rc("legend", fontsize=5)
plt.rc("legend", title_fontsize=6)
plt.rc("font", size=6)
plt.rc("figure", titlesize=7)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["figure.figsize"] = (7.2, 5.0)
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300

fig_width, fig_height, fig_dpi = 7.2, 5.0, 300

gene_type_pretty = {
    "lncRNA": "lncRNA",
    "protein_coding": "protein-coding",
    "pseudogene": "pseudogene",
    "sRNA": "sRNA",
    "N_ambiguous": "ambiguous strand",
    "N_multimapping": "multimapping",
    "N_noFeature": "unannotated",
    "N_unmapped": "unmapped",
}
gene_type_palette = {
    "lncRNA": sns.color_palette("Paired", 12)[5],
    "protein_coding": sns.color_palette("Paired", 12)[1],
    "pseudogene": sns.color_palette("Paired", 12)[3],
    "sRNA": sns.color_palette("Paired", 12)[7],
    "N_ambiguous": "grey",
    "N_multimapping": "grey",
    "N_noFeature": "grey",
    "N_unmapped": "grey",
}
version_palette = {
    "v10": "#bdbdbd",
    "v11": "#525252",
}
marker_map = {"sRNA": "^", "pseudogene": "v", "protein_coding": "s", "lncRNA": "o"}
change_pretty = {
    "new_gene": "new gene",
    "new_transcript": "new transcript",
    "modified_transcript": "modified transcript",
    "modified_gene": "modified gene",
    "reassigned_gene": "reassigned gene",
    "no_change": "no change",
    "removed_gene": "removed gene",
    "unknown": "unknown",
}
lnc_change_order = ["removed_gene", "no_change", "modified_gene", "reassigned_gene", "new_gene"]
pc_amb_bin_order = [
    "no change", "+(0%-1%]", "+(1%-5%]", "+(5%-10%]", "+(10%-100%]",
]
biotype_order_b = ["protein_coding", "lncRNA", "pseudogene", "sRNA"]

annot_cols = [
    "gene_id_stable", "gene_type_grouped", "change_level",
]


## Paths

In [ ]:
gencode_class_path = Path("path/to/gencode_v39_v47_gene_classification.txt")
tx_version_counts_path = Path("path/to/gencode_biotype_transcript_counts.txt")

gtex_colors_json = Path("path/to/gtex_colors.json")
v10_star_agg = Path("path/to/v10_star_agg")
v11_star_agg = Path("path/to/v11_star_agg")

fig_dir = Path("figures")
fig_pdf = fig_dir / "supp_figure_1_quantification_ambiguity.pdf"
fig_png = fig_dir / "supp_figure_1_quantification_ambiguity.png"


## Load data


### GENCODE classification


In [ ]:
gencode_genes = pd.read_table(gencode_class_path, low_memory=False)
print(f"GENCODE genes: {len(gencode_genes):,}")
print(gencode_genes["gene_type_grouped"].value_counts().to_string())


### Tissue file names


In [ ]:
with open(gtex_colors_json) as f:
    tissue_df = pd.DataFrame.from_dict(json.load(f), orient="index")
tissue_df = tissue_df.reset_index(names="tissue_name")
tissue_df["tissue_file_name"] = (
    tissue_df["tissue_name"]
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace(" - ", "_")
    .str.replace(" ", "_")
)
print(f"Tissues: {len(tissue_df):,}")


### Compute — STAR tissue percent reads


In [ ]:
star_agg_by_label = {"v10": v10_star_agg, "v11": v11_star_agg}
star_percent_by_label = {}

for label, star_agg_dir in star_agg_by_label.items():
    all_parts = []
    for tissue_id in tqdm(tissue_df["tissue_file_name"], desc=f"{label} STAR percents"):
        path = star_agg_dir / f"{tissue_id}.ReadsPerGene.out.tab.gz"
        if not path.is_file():
            raise FileNotFoundError(f"Missing STAR ReadsPerGene matrix: {path}")
        tissue_star = pd.read_table(path, low_memory=False)
        sample_cols = [c for c in tissue_star.columns if c.startswith("GTEX-")]
        if not sample_cols:
            raise ValueError(f"No GTEX sample columns in {path}")
        denom = tissue_star[sample_cols].sum(axis=0)
        denom_pct = denom.where(denom != 0, other=pd.NA)
        s = tissue_star[sample_cols].div(denom_pct, axis=1).mean(axis=1)
        all_parts.append(pd.DataFrame({
            "gene_id": tissue_star["gene_id"],
            f"{tissue_id}.percent_reads": s,
        }).set_index("gene_id"))

    percent_reads_df = pd.concat(all_parts, axis=1, join="outer").reset_index()
    percent_reads_df["gene_id"] = percent_reads_df["gene_id"].str.split(".", expand=True)[0]
    percent_reads_df = percent_reads_df.merge(
        gencode_genes[annot_cols],
        left_on="gene_id",
        right_on="gene_id_stable",
        how="left",
    )
    percent_reads_df["gene_type_star"] = percent_reads_df["gene_type_grouped"].fillna(
        percent_reads_df["gene_id"]
    )
    star_percent_by_label[label] = percent_reads_df

v10_star_percent_df = star_percent_by_label["v10"]
v11_star_percent_df = star_percent_by_label["v11"]


### STAR per-tissue percent reads by gene (v10 and v11)

In [ ]:
print(f"v10 STAR rows: {len(v10_star_percent_df):,}; v11 STAR rows: {len(v11_star_percent_df):,}")


### Compute — STAR sample totals


In [ ]:
star_agg_by_label = {"v10": v10_star_agg, "v11": v11_star_agg}
star_total_rows = []
for label, star_agg_dir in star_agg_by_label.items():
    for tissue_id in tqdm(tissue_df["tissue_file_name"], desc=f"{label} STAR totals"):
        path = star_agg_dir / f"{tissue_id}.ReadsPerGene.out.tab.gz"
        if not path.is_file():
            raise FileNotFoundError(f"Missing STAR ReadsPerGene matrix: {path}")
        tissue_star = pd.read_table(path, low_memory=False)
        sample_cols = [c for c in tissue_star.columns if c.startswith("GTEX-")]
        if not sample_cols:
            raise ValueError(f"No GTEX sample columns in {path}")
        denom = tissue_star[sample_cols].sum(axis=0)
        part_tot = denom.rename("total_reads").reset_index()
        part_tot = part_tot.rename(columns={part_tot.columns[0]: "SAMPID"})
        part_tot["tissue_id"] = tissue_id
        part_tot["version"] = label
        star_total_rows.append(part_tot)

star_sample_totals = pd.concat(star_total_rows, ignore_index=True)
print(f"STAR sample totals: {len(star_sample_totals):,} rows")


### STAR per-sample total reads

In [ ]:
for ver, g in star_sample_totals.groupby("version"):
    print(f"  {ver}: n={len(g):,}, median total_reads={g['total_reads'].median():,.0f}")


### Transcript counts over GTEx releases


In [ ]:
tx_version_counts_wide = pd.read_csv(tx_version_counts_path)
tx_version_counts = tx_version_counts_wide.melt(
    id_vars=["gtex_year", "gencode_year", "gtex_version", "gencode_version", "total_transcripts"],
    value_vars=["protein_coding", "lncRNA", "sRNA", "pseudogene"],
    var_name="gene_type",
    value_name="n_transcripts",
)
print(f"Transcript count rows: {len(tx_version_counts):,}")


## Run analysis


### STAR absolute and delta read fractions (panels c–e)


In [ ]:
tissue_cols_v10 = [c for c in v10_star_percent_df.columns if c.endswith(".percent_reads")]
tissue_cols_v11 = [c for c in v11_star_percent_df.columns if c.endswith(".percent_reads")]

grouped_v10 = v10_star_percent_df.groupby("gene_type_star", as_index=False)[tissue_cols_v10].sum()
grouped_v11 = v11_star_percent_df.groupby("gene_type_star", as_index=False)[tissue_cols_v11].sum()

melt_v10 = (
    grouped_v10.melt(id_vars=["gene_type_star"], value_vars=tissue_cols_v10,
                     var_name="tissue_col", value_name="percent_reads")
    .assign(
        tissue_id=lambda d: d["tissue_col"].str.replace(".percent_reads", "", regex=False),
        version="v10",
    )
    .drop(columns=["tissue_col"])
)
melt_v11 = (
    grouped_v11.melt(id_vars=["gene_type_star"], value_vars=tissue_cols_v11,
                     var_name="tissue_col", value_name="percent_reads")
    .assign(
        tissue_id=lambda d: d["tissue_col"].str.replace(".percent_reads", "", regex=False),
        version="v11",
    )
    .drop(columns=["tissue_col"])
)

star_levels = pd.concat([melt_v10, melt_v11], ignore_index=True)
star_levels["percent_reads_pct"] = 100 * star_levels["percent_reads"]

print(f"star_levels rows: {len(star_levels):,}; gene types: {sorted(star_levels['gene_type_star'].unique())}")


### lncRNA % reads by change level (panel d)


In [ ]:
# lncRNA % reads by change_level: tissue-level v11 - v10 deltas
lnc_v11 = v11_star_percent_df[v11_star_percent_df["gene_type_grouped"] == "lncRNA"].copy()
lnc_v10 = v10_star_percent_df[v10_star_percent_df["gene_type_grouped"] == "lncRNA"].copy()
lnc_v11["change_level"] = lnc_v11["change_level"].fillna("unknown")
lnc_v10["change_level"] = lnc_v10["change_level"].fillna("unknown")
# Methods five categories; map transcript-level change_level values into gene-level bins.
_lnc_pool = {
    "modified_transcript": "modified_gene",
    "new_transcript": "modified_gene",
}
lnc_v11["change_level_plot"] = lnc_v11["change_level"].replace(_lnc_pool)
lnc_v10["change_level_plot"] = lnc_v10["change_level"].replace(_lnc_pool)
common_tissue_cols = sorted(set(c for c in lnc_v11.columns if c.endswith(".percent_reads"))
                            & set(c for c in lnc_v10.columns if c.endswith(".percent_reads")))

lnc_sums_v11 = lnc_v11.groupby("change_level_plot")[common_tissue_cols].sum()
lnc_sums_v10 = lnc_v10.groupby("change_level_plot")[common_tissue_cols].sum()
all_change_levels = sorted(set(lnc_sums_v11.index).union(lnc_sums_v10.index))
cl_order = [c for c in lnc_change_order if c in all_change_levels] + [
    c for c in all_change_levels if c not in lnc_change_order
]

shift_rows = []
for cl in cl_order:
    for tissue in common_tissue_cols:
        v11_val = lnc_sums_v11.loc[cl, tissue] if cl in lnc_sums_v11.index else 0.0
        v10_val = lnc_sums_v10.loc[cl, tissue] if cl in lnc_sums_v10.index else 0.0
        shift_rows.append({
            "category": cl,
            "tissue": tissue.split(".")[0],
            "delta": v11_val - v10_val,
        })

star_shift = pd.DataFrame(shift_rows)
star_shift["delta_pct"] = 100 * star_shift["delta"]

print(f"star_shift rows: {len(star_shift):,}")


### Protein-coding % reads by ambiguity gain (panel e)


In [ ]:
# Protein-coding % reads by increase in gene ambiguity (v47 vs v39), pp
# Bins: no change, +(0%-1%], +(1%-5%], +(5%-10%], +(10%-100%]. Ambiguity-loss genes excluded.
pc_v10 = v10_star_percent_df[v10_star_percent_df["gene_type_grouped"] == "protein_coding"].copy()
pc_v11 = v11_star_percent_df[v11_star_percent_df["gene_type_grouped"] == "protein_coding"].copy()
pc_tissue_cols = sorted(
    set(c for c in pc_v10.columns if c.endswith(".percent_reads"))
    & set(c for c in pc_v11.columns if c.endswith(".percent_reads"))
)

pc_genes = gencode_genes[
    (gencode_genes["in_v47_full"] == True)
    & (gencode_genes["gene_type_grouped"] == "protein_coding")
][["gene_id_stable", "pct_ambiguous_v47", "pct_ambiguous_v39"]].copy()
pc_genes[["pct_ambiguous_v47", "pct_ambiguous_v39"]] = (
    pc_genes[["pct_ambiguous_v47", "pct_ambiguous_v39"]].fillna(0)
)
pc_genes["ambig_gain_pp"] = 100 * (
    pc_genes["pct_ambiguous_v47"] - pc_genes["pct_ambiguous_v39"]
)
pc_genes["amb_bin"] = np.select(
    [
        pc_genes["ambig_gain_pp"] == 0,
        (pc_genes["ambig_gain_pp"] > 0) & (pc_genes["ambig_gain_pp"] <= 1),
        (pc_genes["ambig_gain_pp"] > 1) & (pc_genes["ambig_gain_pp"] <= 5),
        (pc_genes["ambig_gain_pp"] > 5) & (pc_genes["ambig_gain_pp"] <= 10),
        (pc_genes["ambig_gain_pp"] > 10) & (pc_genes["ambig_gain_pp"] <= 100),
    ],
    pc_amb_bin_order,
    default=None,
)
pc_genes_binned = pc_genes.dropna(subset=["amb_bin"]).copy()
print(f"protein-coding genes binned (excl. ambig loss): {len(pc_genes_binned):,}")

bin_col = pc_genes_binned[["gene_id_stable", "amb_bin"]].drop_duplicates("gene_id_stable")
pc_v10_binned = pc_v10[["gene_id_stable"] + pc_tissue_cols].merge(bin_col, on="gene_id_stable", how="inner")
pc_v11_binned = pc_v11[["gene_id_stable"] + pc_tissue_cols].merge(bin_col, on="gene_id_stable", how="inner")
pc_sums_v10 = pc_v10_binned.groupby("amb_bin", observed=True)[pc_tissue_cols].sum()
pc_sums_v11 = pc_v11_binned.groupby("amb_bin", observed=True)[pc_tissue_cols].sum()

pc_amb_rows = []
for b in pc_amb_bin_order:
    for tissue in pc_tissue_cols:
        v10_val = pc_sums_v10.loc[b, tissue] if b in pc_sums_v10.index else 0.0
        v11_val = pc_sums_v11.loc[b, tissue] if b in pc_sums_v11.index else 0.0
        pc_amb_rows.append({
            "category": b,
            "tissue": tissue.split(".")[0],
            "delta": v11_val - v10_val,
        })
pc_amb_shift = pd.DataFrame(pc_amb_rows)
pc_amb_shift["delta_pct"] = 100 * pc_amb_shift["delta"]
print(f"pc_amb_shift rows: {len(pc_amb_shift):,}")


### Reassigned / modified detail for panel b


In [ ]:
# Panel b: reassigned / modified gene–transcript–exon detail, stacked by biotype
gc = gencode_genes.loc[gencode_genes["gene_type_grouped"] != "artifact"].copy()
# Classification file splits lncRNA into novel/existing; panel b stacks pooled lncRNA.
gc["gene_type_grouped"] = gc["gene_type_grouped"].replace(
    {"novel_lncRNA": "lncRNA", "existing_lncRNA": "lncRNA"}
)

new_id = gc[(gc["in_v47_full"] == True) & (gc["in_v39_full"] == False)].copy()

# Modified genes (stable ID in both versions with transcript/exon edits)
mod = gc.loc[
    gc["change_level"].isin(["modified_gene", "modified_transcript", "new_transcript"])
].copy()

def _biotype_counts(df, label_col, label_order):
    """Return DataFrame index=label, columns=biotype with gene counts."""
    if label_col not in df.columns or len(df) == 0:
        return pd.DataFrame(0, index=label_order, columns=biotype_order_b)
    ct = pd.crosstab(df[label_col], df["gene_type_grouped"])
    return ct.reindex(index=label_order, columns=biotype_order_b).fillna(0).astype(int)


# New-ID reassignment -> renamed gene / transcript / exon
re_sub_genes = new_id.loc[new_id["change_level"] == "reassigned_gene"].copy()
re_sub_genes["label"] = re_sub_genes["reassignment_subcase"].map({
    "gene_coord_match": "renamed gene",
    "all_new_tx_some_reassigned_tx": "renamed transcript",
    "new_gene_new_coords_some_reassigned_ex": "renamed exon",
})

# Transcript/exon status within modified-gene pool; reassigned_* folded into renamed
tx_order = ["removed transcript", "renamed transcript", "moved transcript", "new transcript"]
ex_order = ["removed exon", "renamed exon", "moved exon", "new exon"]
tx_status_to_label = {
    "removed_transcript": "removed transcript",
    "reassigned_transcript": "renamed transcript",
    "updated_transcript": "moved transcript",
    "new_transcript": "new transcript",
}
ex_status_to_label = {
    "removed_exon": "removed exon",
    "reassigned_exon": "renamed exon",
    "updated_exon": "moved exon",
    "new_exon": "new exon",
}

mod_tx = mod.copy()
mod_tx["label"] = mod_tx["transcript_status"].map(tx_status_to_label)
mod_tx = mod_tx.dropna(subset=["label"])
mod_tx_by_biotype = _biotype_counts(mod_tx, "label", tx_order)

mod_ex = mod.copy()
mod_ex["label"] = mod_ex["exon_status"].map(ex_status_to_label)
mod_ex = mod_ex.dropna(subset=["label"])
mod_ex_by_biotype = _biotype_counts(mod_ex, "label", ex_order)

# Add new-ID renamed transcript/exon into the renamed bars
re_tx = re_sub_genes.loc[re_sub_genes["label"] == "renamed transcript"]
re_ex = re_sub_genes.loc[re_sub_genes["label"] == "renamed exon"]
mod_tx_by_biotype.loc["renamed transcript"] = (
    mod_tx_by_biotype.loc["renamed transcript"]
    + _biotype_counts(re_tx, "label", ["renamed transcript"]).loc["renamed transcript"]
)
mod_ex_by_biotype.loc["renamed exon"] = (
    mod_ex_by_biotype.loc["renamed exon"]
    + _biotype_counts(re_ex, "label", ["renamed exon"]).loc["renamed exon"]
)

gene_by_biotype = _biotype_counts(
    re_sub_genes.loc[re_sub_genes["label"] == "renamed gene"], "label", ["renamed gene"]
)

# Panel-b row order within sections: removed -> renamed -> moved -> new
b_stack_rows = [("gene", "renamed gene", gene_by_biotype)]
b_stack_rows += [("transcript", lab, mod_tx_by_biotype) for lab in tx_order]
b_stack_rows += [("exon", lab, mod_ex_by_biotype) for lab in ex_order]

print(f"panel b rows: {len(b_stack_rows):,}")


## Make figure


In [ ]:
fig = plt.figure(figsize=(fig_width, fig_height), dpi=fig_dpi)
gs = fig.add_gridspec(2, 1, height_ratios=[1, 1.5], hspace=0.5)
# Top: a (transcript counts) | b (reassigned / modified detail)
gs_top = gs[0].subgridspec(1, 2, width_ratios=[1.15, 1.0], wspace=0.6)
ax_a = fig.add_subplot(gs_top[0])
ax_b = fig.add_subplot(gs_top[1])
lines, labels = [], []
for gene_type in ["protein_coding", "lncRNA", "pseudogene", "sRNA"]:
    sub = tx_version_counts[tx_version_counts["gene_type"] == gene_type]
    ln, = ax_a.plot(
        sub["gtex_year"], sub["n_transcripts"],
        marker=marker_map[gene_type], ms=4,
        color=gene_type_palette[gene_type],
        markerfacecolor=gene_type_palette[gene_type],
        markeredgecolor="k", markeredgewidth=0.35,
        label=gene_type_pretty[gene_type],
    )
    lines.append(ln)
    labels.append(gene_type_pretty[gene_type])
ax_a.set_xlabel("GTEx release year")
ax_a.set_ylabel("Annotated transcripts")
ax_a.set_xticks(np.arange(2012, 2026, 3))
ax_a.spines[["top", "right"]].set_visible(False)
ax_a.axvspan(2024, 2025, color="grey", alpha=0.2, zorder=-1, linewidth=0)
ax_a.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax_a.set_ylim(0, 220000)
ax_a.set_yticks([0, 50000, 100000, 150000, 200000])
ax_a.set_yticklabels([0, "50k", "100k", "150k", "200k"])
ax_a.text(
    2024.5, ax_a.get_ylim()[1] * 0.94, r"$v_{10} \rightarrow v_{11}$",
    ha="center", va="bottom", color="grey",
)
ax_a.legend(
    lines, labels, loc="upper left", bbox_to_anchor=(0, 1.15),
    title="Biotype", frameon=False, markerscale=0.6,
)
plot_labels, plot_groups, plot_counts = [], [], []
for group, label, cdf in b_stack_rows:
    plot_labels.append(label)
    plot_groups.append(group)
    plot_counts.append(cdf.loc[label].reindex(biotype_order_b).fillna(0).to_numpy(dtype=float))
plot_counts = np.vstack(plot_counts)
gap = 0.6
group_order = ["exon", "transcript", "gene"]
ys, ylabels, ygroups, ycounts = [], [], [], []
y_cursor = 0.0
for gi, group in enumerate(group_order):
    if gi > 0:
        y_cursor += gap
    idxs = [i for i, g in enumerate(plot_groups) if g == group]
    for i in reversed(idxs):
        ys.append(y_cursor)
        ylabels.append(plot_labels[i])
        ygroups.append(plot_groups[i])
        ycounts.append(plot_counts[i])
        y_cursor += 1.0
ys = np.array(ys)
ycounts = np.vstack(ycounts)
left = np.zeros(len(ys))
for j, gt in enumerate(biotype_order_b):
    vals = ycounts[:, j]
    ax_b.barh(
        ys, vals, left=left, height=0.8,
        color=gene_type_palette[gt], edgecolor="k", linewidth=0.3,
        label=gene_type_pretty[gt],
    )
    left += vals
ax_b.set_yticks(ys)
ax_b.set_yticklabels(ylabels)
ax_b.set_xlabel("Annotated genes")
ax_b.set_ylabel(r"$v_{10} \rightarrow v_{11}$ annotation change")
ax_b.legend(frameon=False, loc="upper right", title="v47 biotype", bbox_to_anchor=(1, 1.15))
ax_b.get_xaxis().set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))
sns.despine(ax=ax_b)
gs_mid = gs[1].subgridspec(1, 2, width_ratios=[1.15, 1.0], wspace=0.6)
ax_c = fig.add_subplot(gs_mid[0])
gs_de = gs_mid[1].subgridspec(2, 1, hspace=0.75)
ax_d = fig.add_subplot(gs_de[0])
ax_e = fig.add_subplot(gs_de[1])
x_label = "% reads / tissue"
possible_types = [
    "protein_coding", "lncRNA", "pseudogene", "sRNA",
    "N_ambiguous", "N_multimapping", "N_noFeature", "N_unmapped",
]
present_types = [t for t in possible_types if t in set(star_levels["gene_type_star"])]
star_plot = star_levels[star_levels["gene_type_star"].isin(present_types)].copy()
type_medians = (
    star_plot.groupby("gene_type_star")["percent_reads_pct"]
    .median()
    .sort_values()
)
plot_types = [t for t in type_medians.index if t in present_types]
sns.boxplot(
    data=star_plot, y="gene_type_star", x="percent_reads_pct", hue="version",
    order=plot_types, hue_order=["v10", "v11"], palette=version_palette,
    showfliers=False, linewidth=0.5, width=0.7, ax=ax_c, orient="h",
)
ax_c.set_yticklabels([gene_type_pretty.get(t, t) for t in plot_types])
ax_c.set_ylabel("")
ax_c.set_xlabel(x_label)
handles, labels = ax_c.get_legend_handles_labels()
labels = [
    l.replace("v10", r"$\mathdefault{v_{10}}$").replace("v11", r"$\mathdefault{v_{11}}$")
    for l in labels
]
ax_c.legend(
    handles, labels, frameon=False, title="GTEx version", loc="upper right",
    handletextpad=0.8, title_fontsize=None, fontsize=None
)
sns.despine(ax=ax_c)
lnc_shift = star_shift.copy()
lnc_shift["label"] = lnc_shift["category"].map(lambda c: change_pretty.get(c, c.replace("_", " ")))
lnc_order_keys = [c for c in lnc_change_order if c in set(lnc_shift["category"])]
lnc_order = [change_pretty.get(c, c.replace("_", " ")) for c in lnc_order_keys]
extra = [
    lab for lab in lnc_shift["label"].unique()
    if lab not in lnc_order
]
lnc_order = lnc_order + extra
sns.boxplot(
    data=lnc_shift, y="label", x="delta_pct",
    color=gene_type_palette["lncRNA"],
    legend=False, showfliers=False, ax=ax_d, linewidth=0.5, order=lnc_order,
)
ax_d.axvline(0, color="k", lw=1, ls="--", zorder=0)
ax_d.set_xlabel(r"lncRNA % reads / tissue $v_{10} \rightarrow v_{11}$")
ax_d.set_ylabel(r"$v_{10} \rightarrow v_{11}$ annotation")
sns.despine(ax=ax_d)
sns.boxplot(
    data=pc_amb_shift, y="category", x="delta_pct",
    color=gene_type_palette["protein_coding"],
    legend=False, showfliers=False, ax=ax_e, linewidth=0.5, order=pc_amb_bin_order,
)
ax_e.axvline(0, color="k", lw=1, ls="--", zorder=0)
ax_e.set_xlabel(r"protein-coding % reads / tissue $v_{10} \rightarrow v_{11}$")
ax_e.set_ylabel(r"$v_{10} \rightarrow v_{11}$ ambiguity")
sns.despine(ax=ax_e)
fig.canvas.draw()
_inv = fig.transFigure.inverted()
_renderer = fig.canvas.get_renderer()
_x_left = min(_inv.transform((ax.get_tightbbox(_renderer).x0, 0))[0] for ax in (ax_a, ax_c))
_x_right = min(_inv.transform((ax.get_tightbbox(_renderer).x0, 0))[0] for ax in (ax_b, ax_d, ax_e))
for ax, label in [(ax_a, "a"), (ax_b, "b"), (ax_c, "c"), (ax_d, "d"), (ax_e, "e")]:
    _y = _inv.transform(ax.transAxes.transform((0, 1.15)))[1]
    _x = _x_left if label in ("a", "c") else _x_right
    fig.text(_x, _y, label, fontweight="bold", va="bottom", ha="right", fontsize=9)
plt.show()


## Save figure


In [ ]:
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_pdf, bbox_inches="tight")
fig.savefig(fig_png, bbox_inches="tight")
print(f"Wrote {fig_pdf}")
print(f"Wrote {fig_png}")


## Print stats


In [ ]:
# Numbers cited for Supplementary Fig. 1 (caption a + Results); existing-lncRNA gene count is printed in Fig. 1.

# Caption a — lncRNA transcripts in panel a (GENCODE refs for GTEx v10/v11)
lnc_tx_by_gtex = (
    tx_version_counts_wide.set_index("gtex_version")["lncRNA"].astype(int)
)
lnc_tx_v10 = int(lnc_tx_by_gtex["v10"])
lnc_tx_v11 = int(lnc_tx_by_gtex["v11"])
print(
    f"Caption a: lncRNA transcripts increase from {lnc_tx_v10:,} in v10 "
    f"to {lnc_tx_v11:,} in v11"
)

# Manuscript — median % reads to lncRNA (Supplementary Fig. 1c,d)
lnc_levels = star_levels[star_levels["gene_type_star"] == "lncRNA"].copy()
lnc_pct_v10 = float(lnc_levels.loc[lnc_levels["version"] == "v10", "percent_reads_pct"].median())
lnc_pct_v11 = float(lnc_levels.loc[lnc_levels["version"] == "v11", "percent_reads_pct"].median())
print(
    f"Supp Fig. 1c,d: median % reads to lncRNA genes: "
    f"{lnc_pct_v10:.2f}% (v10) → {lnc_pct_v11:.2f}% (v11)"
)

# Manuscript — additional lncRNA reads for the median sample.
# STAR GeneCounts totals are PE fragments (same denom as percent_reads); GTEx is paired-end
# with no unpaired reads, so reads = 2 × fragments (matches RNA-SeQC "Total Reads").
use = star_sample_totals[star_sample_totals["version"] == "v11"]
median_sample_frags = float(use["total_reads"].median())
median_sample_reads = 2.0 * median_sample_frags
delta_frac = (lnc_pct_v11 - lnc_pct_v10) / 100
median_additional_lnc_frags = median_sample_frags * delta_frac
median_additional_lnc_reads = median_sample_reads * delta_frac
print(
    f"Supp Fig. 1c,d: median additional lncRNA reads / sample ≈ {median_additional_lnc_reads:,.0f} "
    f"(Δ median lncRNA % × median sample reads {median_sample_reads:,.0f}; "
    f"= 2 × STAR fragments {median_sample_frags:,.0f})"
)
